In [ ]:
!pip install litellm

In [ ]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

In [ ]:
LITE_MODE = True

username = "md-shahnawaj"

dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

from pricer.items import Item

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
items[2].id

In [ ]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [ ]:

SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
print(items[0].full)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# print(os.getenv("GROQ_API_KEY"))ZZ

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]

response = completion(messages=messages, model="LLaMA 3 7OB")

print(response.choices[0].message.content)
print()

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

In [ ]:
MODEL ="LLaMA 3 7OB"

In [ ]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}]}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [ ]:
items[0]

In [ ]:
make_jsonl(items[0])

In [ ]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [ ]:

with open("jsonl/0_1000.jsonl", "rb") as f:
    response = client.files.create(file=f, purpose="batch")
response

In [ ]:
file_id = response.id
file_id

In [ ]:
response = client.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [ ]:
result = client.batches.retrieve(response.id)
print(result.status)

In [ ]:
result = client.batches.retrieve(response.id)
result

In [ ]:
# result = client.batches.retrieve(response.id)
# print(result.status)
# print(result.output_file_id)

In [ ]:
response = client.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary

In [ ]:
print(items[0].full)

In [ ]:
print(items[1000].summary)

In [ ]:
print(items[0].id)

In [ ]:
clean_items = []

for item in items:
    if (
        item.full is not None
        and isinstance(item.full, str)
    ):
        text = item.full.strip()

        # basic filters
        if len(text) < 20:
            continue
        if len(text) > 2000:   # 🔥 strong limit
            continue

        # remove bad patterns
        if "```" in text:
            continue
        if text.count("{") != text.count("}"):
            continue
        if text.count("[") != text.count("]"):
            continue

        # clean text
        text = text.replace("\n", " ").replace("\r", " ")

        item.full = text
        clean_items.append(item)

print(len(clean_items))

In [ ]:
fixed_items = []

for i, item in enumerate(clean_items):
    fixed_items.append({
        "custom_id": str(i),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o-mini",
            "messages": [
                {
                    "role": "user",
                    "content": item.full
                }
            ]
        }
    })

In [ ]:
Batch.create(clean_items, LITE_MODE)

In [ ]:


Batch.run()

In [ ]:
from openai import OpenAI
import os
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


In [ ]:
for b in Batch.batches[:22]:
    res = client.batches.retrieve(b.batch_id)
    print(res.status)

In [ ]:
for b in Batch.batches:
    res = client.batches.retrieve(b.batch_id)
    print(res.status, b.filename)

In [ ]:
for b in Batch.batches[:5]:
    res = client.batches.retrieve(b.batch_id)
    print(res.status)
    print(res.output_file_id)

In [ ]:
Batch.fetch()

In [ ]:
for i, item in enumerate(items):
    if not item.summary:
        print(i)

In [ ]:
for item in items:
    item.full = None
    item.id = None

In [ ]:
# def clean_item(item):
#     item.summary = item.summary or ""
#     item.full = item.full or ""
#     item.prompt = item.prompt or ""
#     item.id = item.id or ""
#     return item

# train = [clean_item(i) for i in train]
# val = [clean_item(i) for i in val]
# test = [clean_item(i) for i in test]

In [ ]:

username = "md-shahnawaj"

# 👇 NEW dataset names (safe — overwrite नहीं होगा)
full = f"{username}/items_full_clean_v2"
lite = f"{username}/items_lite_clean_v2"

# 👇 Toggle
LITE_MODE = True   # True = lite upload | False = full upload

# =========================
# 📦 DATA SPLIT + UPLOAD
# =========================

if LITE_MODE:
    # 🔹 Lite dataset
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]

    print(f"Uploading LITE dataset: {lite}")
    Item.push_to_hub(lite, train, val, test)

else:
    # 🔹 Full dataset
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]

    print(f"Uploading FULL dataset: {full}")
    Item.push_to_hub(full, train, val, test)

    # 🔹 Also create lite version from full (optional but useful)
    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]

    print(f"Uploading LITE version from full: {lite}")
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)